# Qwen3.5-4B A2D BD3LM SFT Training (Colab)
Google Colab A100 üzerinde Qwen3.5-4B diffusion model SFT eğitimi

## 1. Setup & Dependencies

In [ ]:
# GPU kontrolü
!nvidia-smi

In [ ]:
# Repo clone & install
!git clone https://github.com/ZHZisZZ/dllm.git
%cd dllm
!pip install -e '.[flash-attn]' 2>&1 | tail -5

## 2. Tokenizer fix (transformers 5.9+ uyumluluğu)
`apply_chat_template` return_dict=False gerekiyor

In [ ]:
%%bash
# Fix 1: apply_chat_template return_dict=False
sed -i 's/tokenize=True, add_generation_prompt=False$/tokenize=True, add_generation_prompt=False, return_dict=False/' dllm/utils/data.py
sed -i 's/tokenize=True, add_generation_prompt=True$/tokenize=True, add_generation_prompt=True, return_dict=False/' dllm/utils/data.py

# Fix 2: Trainer tokenizer -> processing_class
sed -i 's/tokenizer=tokenizer,$/processing_class=tokenizer,/' examples/a2d/bd3lm/sft.py

echo 'Fixes applied!'

## 3. Qwen3.5 A2D Model Desteği Ekleme
Bu dosyalar repo'da yoksa oluşturulmalı

In [ ]:
%%bash
# Qwen3.5 model dosyası kontrol
if [ ! -f dllm/pipelines/a2d/models/qwen3_5/modeling_qwen3_5.py ]; then
    echo 'Qwen3.5 model dosyası bulunamadı!'
    echo 'Lütfen PC den bu dosyaları yükleyin:'
    echo '  - dllm/pipelines/a2d/models/qwen3_5/modeling_qwen3_5.py'
    echo '  - dllm/pipelines/a2d/models/qwen3_5/__init__.py'
    echo '  - dllm/pipelines/a2d/__init__.py (güncel)'
    echo '  - dllm/pipelines/a2d/convert.py (güncel)'
    echo '  - dllm/utils/models.py (güncel)'
else
    echo 'Qwen3.5 model dosyaları mevcut!'
fi

## 4. Model Convert (AR -> Diffusion)
Qwen3.5-4B multimodal modelinden text backbone çıkarılıp A2D formatına dönüştürülür

In [ ]:
!python dllm/pipelines/a2d/convert.py \
    --model_name_or_path Qwen/Qwen3.5-4B \
    --output_dir .models/a2d/Qwen3.5-4B

## 5. SFT Training (BD3LM + LoRA)

In [ ]:
!accelerate launch --num_processes 1 \
    examples/a2d/bd3lm/sft.py \
    --model_name_or_path .models/a2d/Qwen3.5-4B \
    --output_dir .models/a2d/Qwen3.5-4B/bd3lm/alpaca-sft-lora \
    --lora True --r 16 --lora_alpha 32 \
    --max_steps 500 \
    --learning_rate 1e-4 \
    --per_device_train_batch_size 4 \
    --per_device_eval_batch_size 4 \
    --gradient_accumulation_steps 2 \
    --bf16 True \
    --block_size 32 \
    --max_length 256 \
    --logging_steps 10 \
    --save_strategy steps --save_steps 250 \
    --eval_strategy steps --eval_steps 250 \
    --warmup_ratio 0.05 \
    --report_to none

## 6. Test Inference

In [ ]:
import dllm
from peft import PeftModel

class Args:
    model_name_or_path = '.models/a2d/Qwen3.5-4B'

model = dllm.utils.get_model(model_args=Args()).eval()
tokenizer = dllm.utils.get_tokenizer(model_args=Args())

model = PeftModel.from_pretrained(
    model, '.models/a2d/Qwen3.5-4B/bd3lm/alpaca-sft-lora/checkpoint-final'
)
model = model.merge_and_unload()

sampler_config = dllm.core.samplers.BD3LMSamplerConfig(
    steps=128, max_new_tokens=128, block_size=32,
    temperature=0.2, remasking='low_confidence',
)
sampler = dllm.core.samplers.BD3LMSampler(model=model, tokenizer=tokenizer)

prompt = 'What is diffusion language modeling? Explain briefly.'
messages = [[{'role': 'user', 'content': prompt}]]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True)
outputs = sampler.sample(inputs, sampler_config, return_dict=True)

sequences = dllm.utils.sample_trim(tokenizer, outputs.sequences.tolist(), inputs)
print(sequences[0])

## 7. Model İndirme
Eğitilmiş modeli Google Drive'a kaydet veya indir

In [ ]:
# Google Drive'a kaydet
from google.colab import drive
drive.mount('/content/drive')

!cp -r .models/a2d/Qwen3.5-4B/bd3lm/alpaca-sft-lora/checkpoint-final \
    /content/drive/MyDrive/qwen35-4b-sft-lora-final

print('Model Google Drive a kaydedildi!')